# 03 — Ingestão de Dados Macroeconômicos: API SGS / Banco Central

Este notebook consome a **API pública do SGS (Sistema Gerenciador de Séries Temporais)** do Banco Central do Brasil (BCB).

As séries coletadas enriquecem o modelo de score com **variáveis de contexto macroeconômico e de risco sistêmico**, complementando os dados comportamentais do cliente.

## 📊 Séries Coletadas

| Código SGS | Descrição | Uso no Modelo |
|---|---|---|
| `21127` | Inadimplência — Cartão de Crédito Rotativo (PF) | Proxy de risco do mercado de cartões |
| `21128` | Inadimplência — Cartão de Crédito Parcelado (PF) | Proxy de risco de parcelamento |
| `21114` | Inadimplência — Crédito Pessoal Não Consignado | Benchmark de inadimplência livre |
| `21116` | Inadimplência — Crédito Pessoal Consignado (Privado) | Benchmark de inadimplência consignada |
| `29034` | Comprometimento de Renda das Famílias (SFN Total) | Proxy de stress financeiro agregado |
| `29035` | Comprometimento de Renda (exceto habitacional) | Proxy de stress financeiro no consumo |

## 🔗 Endpoint Base
```
https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados?formato=json&dataInicial=DD/MM/AAAA&dataFinal=DD/MM/AAAA
```

---
> **Destino:** Tabela `emissor_a.silver.macro_bacen_sgs` no Unity Catalog

In [ ]:
# %pip install requests pandas

In [ ]:
import requests
import pandas as pd
from datetime import datetime, date
import time

# ─────────────────────────────────────────────
# CONFIGURAÇÕES
# ─────────────────────────────────────────────
DATA_INICIAL = "01/01/2020"
DATA_FINAL   = datetime.today().strftime("%d/%m/%Y")
BASE_URL     = "https://api.bcb.gov.br/dados/serie/bcdata.sgs.{codigo}/dados"

SERIES = {
    21127: "inadimplencia_cartao_rotativo",
    21128: "inadimplencia_cartao_parcelado",
    21114: "inadimplencia_credito_pessoal_nao_consignado",
    21116: "inadimplencia_credito_pessoal_consignado_privado",
    29034: "comprometimento_renda_sfn_total",
    29035: "comprometimento_renda_exceto_habitacional",
}

print(f"Período de coleta: {DATA_INICIAL} → {DATA_FINAL}")
print(f"Séries a coletar: {len(SERIES)}")

In [ ]:
# ─────────────────────────────────────────────
# FUNÇÃO DE INGESTÃO DA API SGS
# ─────────────────────────────────────────────

def fetch_serie_bacen(codigo: int, nome: str, data_inicial: str, data_final: str) -> pd.DataFrame:
    """Consome uma série temporal do SGS/Bacen e retorna um DataFrame padronizado."""
    url = BASE_URL.format(codigo=codigo)
    params = {
        "formato": "json",
        "dataInicial": data_inicial,
        "dataFinal": data_final,
    }
    
    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        dados = response.json()
        
        if not dados:
            print(f"  ⚠️  Série {codigo} ({nome}): sem dados no período.")
            return pd.DataFrame()
        
        df = pd.DataFrame(dados)
        df.columns = ["data", "valor"]
        df["data"]   = pd.to_datetime(df["data"], format="%d/%m/%Y")
        df["valor"]  = pd.to_numeric(df["valor"], errors="coerce")
        df["codigo_sgs"] = codigo
        df["nome_serie"] = nome
        
        print(f"  ✅  Série {codigo} ({nome}): {len(df)} registros coletados.")
        return df
    
    except requests.exceptions.HTTPError as e:
        print(f"  ❌  Série {codigo} ({nome}): erro HTTP {response.status_code} — {e}")
        return pd.DataFrame()
    except Exception as e:
        print(f"  ❌  Série {codigo} ({nome}): erro inesperado — {e}")
        return pd.DataFrame()

In [ ]:
# ─────────────────────────────────────────────
# COLETA DE TODAS AS SÉRIES
# ─────────────────────────────────────────────

print("🔄 Iniciando coleta da API SGS / Banco Central...\n")

frames = []
for codigo, nome in SERIES.items():
    df_serie = fetch_serie_bacen(codigo, nome, DATA_INICIAL, DATA_FINAL)
    if not df_serie.empty:
        frames.append(df_serie)
    time.sleep(0.5)  # Respeita rate limit da API pública

df_raw = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

print(f"\n📦 Total de registros coletados: {len(df_raw)}")
df_raw.head(10)

In [ ]:
# ─────────────────────────────────────────────
# TRANSFORMAÇÃO: FORMATO WIDE (pivot)
# Uma linha por mês, uma coluna por série
# ─────────────────────────────────────────────

df_wide = df_raw.pivot_table(
    index="data",
    columns="nome_serie",
    values="valor",
    aggfunc="last"
).reset_index()

df_wide.columns.name = None
df_wide = df_wide.sort_values("data").reset_index(drop=True)

# Metadados úteis
df_wide["ano"]         = df_wide["data"].dt.year
df_wide["mes"]         = df_wide["data"].dt.month
df_wide["data_carga"]  = pd.Timestamp.today().normalize()
df_wide["fonte"]       = "BCB/SGS"

print(f"Shape final (wide): {df_wide.shape}")
df_wide.tail()

In [ ]:
# ─────────────────────────────────────────────
# ANÁLISE EXPLORATÓRIA RÁPIDA
# ─────────────────────────────────────────────

import matplotlib.pyplot as plt
import matplotlib.dates as mdates

colunas_plot = [
    "inadimplencia_cartao_rotativo",
    "inadimplencia_credito_pessoal_nao_consignado",
    "comprometimento_renda_sfn_total",
]

fig, axes = plt.subplots(len(colunas_plot), 1, figsize=(12, 8), sharex=True)
fig.suptitle("Séries Macroeconômicas — Banco Central (SGS)", fontsize=13, fontweight="bold")

for ax, col in zip(axes, colunas_plot):
    if col in df_wide.columns:
        ax.plot(df_wide["data"], df_wide[col], linewidth=1.8, color="#1565C0")
        ax.set_ylabel("%", fontsize=9)
        ax.set_title(col.replace("_", " ").title(), fontsize=9)
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        ax.grid(axis="y", alpha=0.3)

plt.xlabel("Data")
plt.tight_layout()
plt.show()

In [ ]:
# ─────────────────────────────────────────────
# ENGENHARIA DE FEATURES MACROECONÔMICAS
# Variáveis derivadas úteis para o modelo de score
# ─────────────────────────────────────────────

df_features = df_wide.copy()

# Variação mensal (momentum de inadimplência)
for col in ["inadimplencia_cartao_rotativo", "inadimplencia_credito_pessoal_nao_consignado"]:
    if col in df_features.columns:
        df_features[f"{col}_var_mom"] = df_features[col].pct_change() * 100

# Média móvel 3 meses (suaviza sazonalidade)
for col in ["comprometimento_renda_sfn_total", "comprometimento_renda_exceto_habitacional"]:
    if col in df_features.columns:
        df_features[f"{col}_ma3"] = df_features[col].rolling(3).mean()

# Índice composto de stress macroeconômico (média ponderada das inadimplências)
cols_inadi = [c for c in df_features.columns if c.startswith("inadimplencia_") and not c.endswith("_var_mom")]
if cols_inadi:
    df_features["indice_stress_macro"] = df_features[cols_inadi].mean(axis=1)

print(f"Features geradas: {df_features.shape[1]} colunas")
df_features[["data", "indice_stress_macro", "comprometimento_renda_sfn_total_ma3"]].tail(6)

In [ ]:
# ─────────────────────────────────────────────
# PERSISTÊNCIA NO UNITY CATALOG (Databricks)
# ─────────────────────────────────────────────

# Converte para Spark DataFrame e salva na camada Silver
CATALOG = "emissor_a"
SCHEMA  = "silver"
TABELA  = "macro_bacen_sgs"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA}")

df_spark = spark.createDataFrame(df_features)

(
    df_spark.write
    .format("delta")
    .mode("overwrite")                    # Substitui a tabela a cada carga (idempotente)
    .option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.{SCHEMA}.{TABELA}")
)

print(f"✅ Tabela salva: {CATALOG}.{SCHEMA}.{TABELA}")
spark.sql(f"DESCRIBE TABLE {CATALOG}.{SCHEMA}.{TABELA}").show(truncate=False)

In [ ]:
# ─────────────────────────────────────────────
# VALIDAÇÃO PÓS-CARGA
# ─────────────────────────────────────────────

df_check = spark.sql(f"""
    SELECT
        COUNT(*)        AS total_registros,
        MIN(data)       AS data_mais_antiga,
        MAX(data)       AS data_mais_recente,
        COUNT(DISTINCT ano) AS anos_cobertos
    FROM {CATALOG}.{SCHEMA}.{TABELA}
""")

df_check.show()

## ✅ Próximos Passos

A tabela `emissor_a.silver.macro_bacen_sgs` está disponível para:

1. **`02_feature_engineering/`** — Fazer `JOIN` temporal (por `ano` + `mes`) com os dados de clientes antes do treino
2. **`06_monitoring/`** — Monitorar variações do `indice_stress_macro` para detectar mudanças de regime econômico
3. **`07_agents/`** — O agente pode acionar nova ingestão quando o stress macro superar um threshold

### 🔗 Join com dados de clientes (exemplo)
```sql
SELECT c.*, m.indice_stress_macro, m.comprometimento_renda_sfn_total
FROM emissor_a.gold.clientes c
LEFT JOIN emissor_a.silver.macro_bacen_sgs m
  ON YEAR(c.data_referencia)  = m.ano
 AND MONTH(c.data_referencia) = m.mes
```